# 01. Exploratory Data Analysis, Dimensionality Reduction & Clustering
This notebook demonstrates:
1. Ingestion of IEEE-CIS transaction streams via Polars.
2. Strict out-of-time chronological partitioning without lookahead leakage.
3. IncrementalPCA scree plots & selection of components capturing $\ge 90\%$ variance.
4. MiniBatch K-Means clustering ($K=8$) and distance vector feature augmentation.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from src.data.ingestion import run_ingestion
from src.data.preprocessor import run_preprocessing
from src.eda_baselines.decomposition import PCARepresentationLearner
from src.eda_baselines.clustering import KMeansClusterFeatureGenerator
from src.utils.logger import set_seed

set_seed(42)

In [ ]:
# Load and preprocess data
preprocessor, paths = run_preprocessing()
train_df = pl.read_parquet(paths['train'])
dev_df = pl.read_parquet(paths['dev'])

X_train, y_train, meta_train = preprocessor.transform(train_df)
X_dev, y_dev, meta_dev = preprocessor.transform(dev_df)
print(f"Preprocessed shapes: Train={X_train.shape}, Dev={X_dev.shape}")

In [ ]:
# Fit PCA on train split only
pca = PCARepresentationLearner(variance_threshold=0.90)
pca.fit(X_train)

plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', color='#2563eb')
plt.axhline(0.90, color='r', linestyle='--', label='90% Variance Target')
plt.title('Cumulative Explained Variance Ratio')
plt.xlabel('Number of Components')
plt.ylabel('Variance Explained')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Fit MiniBatchKMeans and compute cluster distances
X_train_pca = pca.transform(X_train)
kmeans = KMeansClusterFeatureGenerator(n_clusters=8)
kmeans.fit(X_train_pca)

dists = kmeans.transform(pca.transform(X_dev))
print(f"Augmented distance vectors shape: {dists.shape}")